In [29]:
import os
import matplotlib.pyplot as plt
import numpy as np
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

path = kagglehub.dataset_download("blastchar/telco-customer-churn")
df = pd.read_csv(os.path.join(path, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))

# print(df.info())
# print(df.isnull().sum())
# print((df == " ").sum())

df["TotalCharges"] = df["TotalCharges"].replace(" ", np.nan)
df["TotalCharges"] = df["TotalCharges"].astype(float)

df.dropna(inplace=True)
df.drop("customerID", axis=1, inplace=True)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

X = df.drop("Churn", axis=1)
y = df["Churn"]

categorical_cols = X.select_dtypes(include="object").columns
numerical_cols = X.select_dtypes(exclude="object").columns

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [30]:
X_train.to_csv("train_features.csv", index=False)
y_train.to_csv("train_labels.csv", index=False)
X_test.to_csv("test_features.csv", index=False)
y_test.to_csv("test_labels.csv", index=False)

In [41]:
from sklearn.pipeline import Pipeline as SK_Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_pipeline = SK_Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_cols),
    ("cat", cat_pipeline, categorical_cols)
])


In [72]:
import mlflow
import mlflow.sklearn
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from sklearn.metrics import f1_score
from imblearn.pipeline import Pipeline as Imb_Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_score, recall_score
import joblib

space = {
    "n_estimators": hp.choice("n_estimators", [100, 200, 300]),
    "max_depth": hp.choice("max_depth", [5, 10, 15, 20]),
    "min_samples_split": hp.choice("min_samples_split", [2, 5, 10]),
    "min_samples_leaf": hp.choice("min_samples_leaf", [1, 2, 4])
}

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("telco_churn")

def objective(params):
    with mlflow.start_run(nested=True):
        classifier = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            min_samples_leaf=params["min_samples_leaf"],
            random_state=42,
            n_jobs=-1
        )
        pipeline = Imb_Pipeline([
            ("preprocessor", preprocessor),
            ("smote", SMOTE(random_state=42)),
            ("classifier", classifier)
        ])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        f1 = f1_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        accuracy = accuracy_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_pred)
        
        mlflow.log_params(params)
        
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("precission", precision)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("roc_auc", roc_auc)

        joblib.dump(final_pipeline, "model.pkl")
        mlflow.log_artifact("model.pkl")
        
        return {"loss": -f1, "status": STATUS_OK}

trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=20,
    trials=trials,
    rstate=np.random.default_rng(42)
)

best_classifier = RandomForestClassifier(
    n_estimators=[100, 200, 300][best["n_estimators"]],
    max_depth=[5, 10, 15, 20][best["max_depth"]],
    min_samples_split=[2, 5, 10][best["min_samples_split"]],
    min_samples_leaf=[1, 2, 4][best["min_samples_leaf"]],
    random_state=42,
    n_jobs=-1
)

final_pipeline = Imb_Pipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", best_classifier)
])

final_pipeline.fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)

f1 = f1_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred)

In [97]:
f1, recall, precision, accuracy, roc_auc

(0.6183115338882283,
 0.6951871657754011,
 0.556745182012848,
 0.7718550106609808,
 0.7473999720454934)

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
import sagemaker
import boto3
from datetime import datetime

session = sagemaker.Session()

role = sagemaker.get_execution_role()

bucket = session.default_bucket()
current_time = datetime.strftime(datetime.now(), "%Y%m%d%H%M%s")

job_name = "telco-preprocessing-" + current_time

processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
)

processor.run(
    code="preprocess.py",
    job_name=job_name
    inputs=[
        ProcessingInput(
            source=f"s3://{bucket}/telco/data/",
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train",
            destination=f"s3://{bucket}/telco/processed/train"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test",
            destination=f"s3://{bucket}/telco/processed/test"
        )
    ]
)

In [33]:

role = "arn:aws:iam::395435558728:role/SagemakerRole"

In [ ]:
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.inputs import TrainingInput
import sagemaker
from datetime import datetime

session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()

preprocessed_training_data = "s3://sagemaker-us-east-1-395435558728/telco/data/train/"
preprocessed_test_data = "s3://sagemaker-us-east-1-395435558728/telco/data/test/"
current_time = datetime.strftime(datetime.now(), "%Y%d%m%H%M%S")
job_name = "telco-training-" + current_time

sklearn = SKLearn(
    entry_point="train.py", 
    framework_version="1.2-1", 
    instance_type="ml.m5.large", 
    role=role, 
    py_version="py3",
    output_path=f"s3://{bucket}/telco/model"
)
sklearn.fit(
    inputs={
        "train": TrainingInput(
            s3_data=f"s3://{bucket}/telco/data/train/",
            content_type="text/csv"
        ),
        "test": TrainingInput(
            s3_data=f"s3://{bucket}/telco/data/test/",
            content_type="text/csv"
        )
    },
    job_name=job_name
)

In [37]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
import sagemaker
import boto3
from datetime import datetime

session = sagemaker.Session()

role = sagemaker.get_execution_role()

bucket = session.default_bucket()
current_time = datetime.strftime(datetime.now(), "%Y%m%d%H%M%s")

job_name = "telco-preprocessing-" + current_time

processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
)

processor.run(
    code="preprocess.py",
    job_name=job_name,
    inputs=[
        ProcessingInput(
            source=f"s3://{bucket}/telco/data/",
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train",
            destination=f"s3://{bucket}/telco/processed/train"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test",
            destination=f"s3://{bucket}/telco/processed/test"
        )
    ]
)

In [ ]:
from sagemaker.sklearn.model import SKLearnModel
import sagemaker

session = sagemaker.Session()
role = sagemaker.get_execution_role()

model = SKLearnModel(
    model_data="s3://sagemaker-us-east-1-395435558728/telco/model/telco-training-20263105152042/output/model.tar.gz",
    role=role,
    framework_version="1.2-1",
    entry_point="inference.py",
    sagemaker_session=session
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="telco-churn-endpoint"
)

In [ ]:
import boto3

sm = boto3.client("sagemaker")

response = sm.create_model_package(
    ModelPackageGroupName="telco-churn-model-group",
    ModelApprovalStatus="PendingManualApproval",
    InferenceSpecification={
        "Containers": [
            {
                "Image": image_uri,
                "ModelDataUrl": model_artifact_s3_uri
            }
        ],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"]
    }
)

model_package_arn = response["ModelPackageArn"]

print(model_package_arn)

In [ ]:
sagemaker_client.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InstanceType": "ml.m5.large",
            "InitialInstanceCount": 1
        }
    ]
)

In [ ]:
import boto3

sm = boto3.client("sagemaker")

response = sm.create_model(
    ModelName="telco-churn-model",
    ExecutionRoleArn=role_arn,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": "s3://sagemaker-us-east-1-395435558728/telco/model/telco-training-20263105152042/output/model.tar.gz",
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "s3://your-bucket/inference/inference.tar.gz"
        }
    }
)

print(response)

In [ ]:
sm.create_endpoint_config(
    EndpointConfigName="telco-endpoint-config",
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": "telco-churn-model",
            "InitialInstanceCount": 1,
            "InstanceType": "ml.m5.large",
            "InitialVariantWeight": 1
        }
    ]
)

In [ ]:
sm.create_endpoint(
    EndpointName="telco-churn-endpoint",
    EndpointConfigName="telco-endpoint-config"
)

In [ ]:
sagemaker_client.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InstanceType": "ml.m5.large",
            "InitialInstanceCount": 1
        }
    ]
)

In [ ]:
sagemaker_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=config_name
)

In [5]:
a = {
  "Error": "States.TaskFailed",
  "Cause": "{\"SdkResponseMetadata\":null,\"SdkHttpMetadata\":null,\"TrainingJobName\":\"sbx-tsp-telco-churn-training-2026-07-01-15-11-29\",\"TrainingJobArn\":\"arn:aws:sagemaker:us-east-1:395435558728:training-job/sbx-tsp-telco-churn-training-2026-07-01-15-11-29\",\"ProcessingJobArn\":null,\"TuningJobArn\":null,\"LabelingJobArn\":null,\"AutoMLJobArn\":null,\"ModelArtifacts\":{\"S3ModelArtifacts\":\"s3://sbx-tsp-telco-churn-s3-workspace/telco-pipeline/model/sbx-tsp-telco-churn-training-2026-07-01-15-11-29/output/model.tar.gz\"},\"TrainingJobOutput\":null,\"TrainingJobStatus\":\"Failed\",\"SecondaryStatus\":\"Failed\",\"FailureReason\":\"AlgorithmError: framework error: \\nTraceback (most recent call last):\\n  File \\\"/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_trainer.py\\\", line 84, in train\\n    entrypoint()\\n  File \\\"/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py\\\", line 39, in main\\n    train(environment.Environment())\\n  File \\\"/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py\\\", line 31, in train\\n    entry_point.run(uri=training_environment.module_dir,\\n  File \\\"/miniconda3/lib/python3.9/site-packages/sagemaker_training/entry_point.py\\\", line 108, in run\\n    return runner.get(runner_type, user_entry_point, args, env_vars, extra_opts).run(\\n  File \\\"/miniconda3/lib/python3.9/site-packages/sagemaker_training/process.py\\\", line 424, in run\\n    process = check_error(\\n  File \\\"/miniconda3/lib/python3.9/site-packages/sagemaker_training/process.py\\\", line 335, in check_error\\n    raise error_class(\\nsagemaker_training.errors.ExecuteUserScriptError: ExecuteUserScriptError:\\nExitCode 1\\nErro\",\"HyperParameters\":{\"sagemaker_container_log_level\":\"20\",\"sagemaker_program\":\"\\\"train.py\\\"\",\"training_job_name\":\"\\\"sbx-tsp-telco-churn-training-2026-07-01-15-11-29\\\"\",\"max_depth\":\"10\",\"min_samples_split\":\"2\",\"sagemaker_region\":\"\\\"us-east-1\\\"\",\"n_estimators\":\"200\",\"workspace_bucket\":\"\\\"sbx-tsp-telco-churn-s3-workspace\\\"\",\"sagemaker_job_name\":\"\\\"sagemaker-scikit-learn-2026-07-01-15-11-29-602\\\"\",\"sagemaker_submit_directory\":\"\\\"s3://sbx-tsp-telco-churn-s3-workspace/sagemaker-scikit-learn-2026-07-01-15-11-29-473/source/sourcedir.tar.gz\\\"\",\"bucket_prefix\":\"\\\"telco-pipeline\\\"\",\"min_samples_leaf\":\"1\"},\"AlgorithmSpecification\":{\"TrainingImage\":\"683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3\",\"AlgorithmName\":null,\"TrainingInputMode\":\"FILE\",\"MetricDefinitions\":null,\"EnableSageMakerMetricsTimeSeries\":null,\"ContainerEntrypoint\":null,\"ContainerArguments\":null,\"TrainingImageConfig\":null},\"RoleArn\":\"arn:aws:iam::395435558728:role/sbx-tsp-telco-churn-sagemaker-exec-role\",\"InputDataConfig\":[{\"ChannelName\":\"train\",\"DataSource\":{\"S3DataSource\":{\"S3DataType\":\"S3_PREFIX\",\"S3Uri\":\"s3://sbx-tsp-telco-churn-s3-workspace/telco-pipeline/data/processed/train\",\"S3DataDistributionType\":\"FULLY_REPLICATED\",\"AttributeNames\":null,\"InstanceGroupNames\":null,\"ModelAccessConfig\":null,\"HubAccessConfig\":null},\"FileSystemDataSource\":null,\"DatasetSource\":null},\"ContentType\":null,\"CompressionType\":\"NONE\",\"RecordWrapperType\":\"NONE\",\"InputMode\":null,\"ShuffleConfig\":null,\"EnableFFM\":null},{\"ChannelName\":\"test\",\"DataSource\":{\"S3DataSource\":{\"S3DataType\":\"S3_PREFIX\",\"S3Uri\":\"s3://sbx-tsp-telco-churn-s3-workspace/telco-pipeline/data/processed/test\",\"S3DataDistributionType\":\"FULLY_REPLICATED\",\"AttributeNames\":null,\"InstanceGroupNames\":null,\"ModelAccessConfig\":null,\"HubAccessConfig\":null},\"FileSystemDataSource\":null,\"DatasetSource\":null},\"ContentType\":null,\"CompressionType\":\"NONE\",\"RecordWrapperType\":\"NONE\",\"InputMode\":null,\"ShuffleConfig\":null,\"EnableFFM\":null}],\"OutputDataConfig\":{\"KmsKeyId\":null,\"S3OutputPath\":\"s3://sbx-tsp-telco-churn-s3-workspace/telco-pipeline/model\",\"CompressionType\":null,\"RemoveJobNameFromS3OutputPath\":null,\"DisableModelUpload\":null,\"Channels\":null},\"ResourceConfig\":{\"InstanceType\":\"ml.m5.large\",\"InstanceCount\":1,\"VolumeSizeInGB\":30,\"VolumeKmsKeyId\":null,\"KeepAlivePeriodInSeconds\":0,\"CapacityReservationIds\":null,\"InstanceGroups\":null,\"CapacitySchedulesConfig\":null,\"TrainingPlanArn\":null,\"UseReservedCapacity\":null,\"InstancePlacementConfig\":null},\"WarmPoolStatus\":null,\"VpcConfig\":null,\"StoppingCondition\":{\"MaxRuntimeInSeconds\":86400,\"MaxWaitTimeInSeconds\":null,\"MaxPendingTimeInSeconds\":null},\"CreationTime\":1782918844558,\"TrainingStartTime\":1782918882668,\"TrainingEndTime\":1782918997481,\"LastModifiedTime\":1782918997481,\"SecondaryStatusTransitions\":[{\"Status\":\"Starting\",\"StartTime\":1782918844558,\"EndTime\":1782918882668,\"StatusMessage\":\"Preparing the instances for training\"},{\"Status\":\"Downloading\",\"StartTime\":1782918882668,\"EndTime\":1782918974122,\"StatusMessage\":\"Downloading the training image\"},{\"Status\":\"Training\",\"StartTime\":1782918974122,\"EndTime\":1782918985002,\"StatusMessage\":\"Training image download completed. Training in progress.\"},{\"Status\":\"Uploading\",\"StartTime\":1782918985002,\"EndTime\":1782918997481,\"StatusMessage\":\"Uploading generated training model\"},{\"Status\":\"Failed\",\"StartTime\":1782918997481,\"EndTime\":1782918997481,\"StatusMessage\":\"Training job failed\"}],\"FinalMetricDataList\":null,\"EnableNetworkIsolation\":null,\"EnableInterContainerTrafficEncryption\":null,\"EnableManagedSpotTraining\":null,\"CheckpointConfig\":null,\"TrainingTimeInSeconds\":null,\"BillableTimeInSeconds\":null,\"BillableTokenCount\":null,\"DebugHookConfig\":{\"LocalPath\":null,\"S3OutputPath\":\"s3://sbx-tsp-telco-churn-s3-workspace/telco-pipeline/model\",\"HookParameters\":null,\"CollectionConfigurations\":null},\"ExperimentConfig\":null,\"DebugRuleConfigurations\":null,\"TensorBoardOutputConfig\":null,\"DebugRuleEvaluationStatuses\":null,\"UpstreamPlatformConfig\":null,\"ProfilerConfig\":null,\"ProfilerRuleConfigurations\":null,\"ProfilerRuleEvaluationStatuses\":null,\"ProfilingStatus\":null,\"Environment\":null,\"RetryStrategy\":null,\"LastModifiedBy\":null,\"CreatedBy\":null,\"DisableEFA\":null,\"ProcessingJobConfig\":null,\"ImageMetadata\":null,\"RemoteDebugConfig\":null,\"ResourceTags\":null,\"InfraCheckConfig\":null,\"ServerlessJobConfig\":null,\"MlflowConfig\":null,\"ModelPackageConfig\":null,\"MlflowDetails\":null,\"ProgressInfo\":null,\"OutputModelPackageArn\":null,\"Tags\":{\"AWS_STEP_FUNCTIONS_EXECUTION_ARN\":\"arn:aws:states:us-east-1:395435558728:execution:sbx-tsp-telco-churn-sfn-state-machine-workflow:8aea682a-442f-4986-b913-b8ec6cd8712c\",\"MANAGED_BY_AWS\":\"STARTED_BY_STEP_FUNCTIONS\"}}"
}

In [6]:
import json
b = json.loads(a["Cause"])

In [7]:
b

{'SdkResponseMetadata': None,
 'SdkHttpMetadata': None,
 'TrainingJobName': 'sbx-tsp-telco-churn-training-2026-07-01-15-11-29',
 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:395435558728:training-job/sbx-tsp-telco-churn-training-2026-07-01-15-11-29',
 'ProcessingJobArn': None,
 'TuningJobArn': None,
 'LabelingJobArn': None,
 'AutoMLJobArn': None,
 'ModelArtifacts': {'S3ModelArtifacts': 's3://sbx-tsp-telco-churn-s3-workspace/telco-pipeline/model/sbx-tsp-telco-churn-training-2026-07-01-15-11-29/output/model.tar.gz'},
 'TrainingJobOutput': None,
 'TrainingJobStatus': 'Failed',
 'SecondaryStatus': 'Failed',
 'FailureReason': 'AlgorithmError: framework error: \nTraceback (most recent call last):\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_trainer.py", line 84, in train\n    entrypoint()\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py", line 39, in main\n    train(environment.Environment())\n  File "/miniconda3/lib/python3

In [9]:
c= 'AlgorithmError: framework error: \nTraceback (most recent call last):\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_trainer.py", line 84, in train\n    entrypoint()\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py", line 39, in main\n    train(environment.Environment())\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py", line 31, in train\n    entry_point.run(uri=training_environment.module_dir,\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_training/entry_point.py", line 108, in run\n    return runner.get(runner_type, user_entry_point, args, env_vars, extra_opts).run(\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_training/process.py", line 424, in run\n    process = check_error(\n  File "/miniconda3/lib/python3.9/site-packages/sagemaker_training/process.py", line 335, in check_error\n    raise error_class(\nsagemaker_training.errors.ExecuteUserScriptError: ExecuteUserScriptError:\nExitCode 1\nErro'

In [11]:
print(c)

AlgorithmError: framework error: 
Traceback (most recent call last):
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_trainer.py", line 84, in train
    entrypoint()
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py", line 39, in main
    train(environment.Environment())
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/training.py", line 31, in train
    entry_point.run(uri=training_environment.module_dir,
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_training/entry_point.py", line 108, in run
    return runner.get(runner_type, user_entry_point, args, env_vars, extra_opts).run(
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_training/process.py", line 424, in run
    process = check_error(
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_training/process.py", line 335, in check_error
    raise error_class(
sagemaker_training.errors.ExecuteUserScriptError: ExecuteUserScri